In [14]:
import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)

3.13.15 2.3.4 2.3.3 1.7.2 3.10.7


In [15]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 2666         # last 4 digits of your student ID, as an integer (0421 -> 421)
data = pd.read_csv("data/spambase.csv")
feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)

(4204, 57) 39.9 (3153, 57) (1051, 57)


In [16]:
from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)
bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, oob_score=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1)
lr = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), LogisticRegression(max_iter=2000))
bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)
knn = make_pipeline(FunctionTransformer(np.log1p), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)
members = [("lr", lr), ("knn", knn), ("tree", tree)]   # members of voting and stacking in Parts 3-4

# 1. Bagging and out-of-bag estimates
A bootstrap sample draws n rows from n with replacement. A row that a tree never saw is out of bag (OOB) for that tree. Here you rebuild scikit-learn's OOB estimate yourself, compare bagged learners with the learners they are made of, and compare three estimates of the same forest.

In [17]:
def oob_mask(sample_indices, n_samples):
    """Boolean mask of the rows that do NOT occur in sample_indices."""
    import numpy as np

    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False   # dtype=int so that an empty list works
    return mask

In [18]:
# the three examples from the task, plus the type/shape and "arguments unchanged" rules
assert oob_mask([0, 2, 2], 5).tolist() == [False, True, False, True, True]
assert oob_mask([], 3).tolist() == [True, True, True]
assert oob_mask(np.array([3, 0]), 4).tolist() == [False, True, True, False]

idx = np.array([3, 0, 3])
idx_before = idx.copy()
m = oob_mask(idx, 4)
assert (idx == idx_before).all() and isinstance(m, np.ndarray) and m.dtype == bool and m.shape == (4,)
print("oob_mask: all examples pass")

oob_mask: all examples pass


### 1.1 Out-of-bag share per tree

In [19]:
bag.fit(X_train, y_train)
n = len(X_train)

oob_shares = np.array([oob_mask(s, n).mean() for s in bag.estimators_samples_])
mean_oob_share = oob_shares.mean()

print(f"n (training rows)                  : {n}")
print(f"mean out-of-bag share over {len(oob_shares)} trees: {mean_oob_share:.4f}")
print(f"mean in-bag share                  : {1 - mean_oob_share:.4f}")
print(f"formula (1 - 1/n)^n                : {(1 - 1 / n) ** n:.4f}")
print(f"formula 1 - (1 - 1/n)^n            : {1 - (1 - 1 / n) ** n:.4f}")

n (training rows)                  : 3153
mean out-of-bag share over 100 trees: 0.3680
mean in-bag share                  : 0.6320
formula (1 - 1/n)^n                : 0.3678
formula 1 - (1 - 1/n)^n            : 0.6322


### 1.2 Out-of-bag accuracy by hand

In [20]:
votes = np.zeros((n, 2), dtype=int)
for tree_k, samples_k in zip(bag.estimators_, bag.estimators_samples_):
    m = oob_mask(samples_k, n)
    pred = tree_k.predict(X_train[m])
    votes[np.flatnonzero(m), pred] += 1          # one vote for the predicted class, only for rows out of bag

oob_pred = votes.argmax(axis=1)                  # a tie goes to class 0
oob_acc_hand = (oob_pred == y_train).mean()
print(f"rows that were out of bag for no tree at all: {(votes.sum(axis=1) == 0).sum()}")
print(f"my out-of-bag accuracy : {oob_acc_hand:.4f}")
print(f"bag.oob_score_         : {bag.oob_score_:.4f}")
print("identical:", np.isclose(oob_acc_hand, bag.oob_score_))

rows that were out of bag for no tree at all: 0
my out-of-bag accuracy : 0.9340
bag.oob_score_         : 0.9340
identical: True


### 1.3 Five models on the same folds

In [21]:
from sklearn.model_selection import cross_validate


def fold_table(scores):
    """Fold scores, mean and std (np.std, ddof=0) of every model in a dict name -> five scores."""
    t = pd.DataFrame({name: s for name, s in scores.items()}).T
    t.columns = [f"fold {i}" for i in range(1, t.shape[1] + 1)]
    t["mean"] = [s.mean() for s in scores.values()]
    t["std"] = [s.std() for s in scores.values()]
    return t


def paired_table(scores, pairs):
    """Paired fold differences A - B for every (A, B) in pairs, with mean, std and the 'inside the fold spread' rule."""
    rows = {}
    for a, b in pairs:
        d = scores[a] - scores[b]
        rows[f"{a} - {b}"] = list(d) + [d.mean(), d.std(), abs(d.mean()) <= d.std()]
    t = pd.DataFrame(rows).T
    t.columns = [f"fold {i}" for i in range(1, len(d) + 1)] + ["mean", "std", "inside spread"]
    return t


five_models = {"tree": tree, "bag": bag, "rf": rf, "lr": lr, "bag_lr": bag_lr}
fold_scores = {name: cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy")["test_score"]
               for name, model in five_models.items()}

display(fold_table(fold_scores).round(4))
print("paired fold differences (accuracy):")
display(paired_table(fold_scores, [("bag", "tree"), ("rf", "tree"), ("bag_lr", "lr")]).round(4))

,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std
tree,0.9113,0.9303,0.9097,0.9000,0.9095,0.9121,0.0099
bag,0.9239,0.9493,0.9334,0.9206,0.9222,0.9299,0.0107
rf,0.9366,0.9556,0.9414,0.9476,0.9508,0.9464,0.0067
lr,0.9319,0.9525,0.9366,0.9333,0.9429,0.9394,0.0075
bag_lr,0.9334,0.9525,0.9366,0.9302,0.9413,0.9388,0.0078


paired fold differences (accuracy):


,fold 1,fold 2,fold 3,fold 4,fold 5,mean,std,inside spread
bag - tree,0.012678,0.019017,0.023772,0.020635,0.012698,0.01776,0.004414,False
rf - tree,0.025357,0.025357,0.031696,0.047619,0.04127,0.03426,0.008862,False
bag_lr - lr,0.001585,0.0,0.0,-0.003175,-0.001587,-0.000635,0.001618,True


### 1.4 One forest, three estimates

In [22]:
rf.fit(X_train, y_train)

n_test = len(y_test)
p_test = rf.score(X_test, y_test)
se_test = np.sqrt(p_test * (1 - p_test) / n_test)
rf_cv_mean, rf_cv_std = fold_scores["rf"].mean(), fold_scores["rf"].std()

print(f"rf.oob_score_                  : {rf.oob_score_:.4f}")
print(f"5-fold CV mean (std over folds): {rf_cv_mean:.4f} ({rf_cv_std:.4f})")
print(f"test accuracy p                : {p_test:.4f}  with standard error sqrt(p(1-p)/n_test) = {se_test:.4f}  (n_test = {n_test})")
print(f"|oob_score_ - CV mean| = {abs(rf.oob_score_ - rf_cv_mean):.4f}  vs  std of the forest's fold scores = {rf_cv_std:.4f}"
      f"  ->  agree: {abs(rf.oob_score_ - rf_cv_mean) <= rf_cv_std}")
print(f"|test accuracy - oob_score_| = {abs(p_test - rf.oob_score_):.4f}  ({abs(p_test - rf.oob_score_) / se_test:.2f} standard errors)")
print(f"|test accuracy - CV mean|    = {abs(p_test - rf_cv_mean):.4f}  ({abs(p_test - rf_cv_mean) / se_test:.2f} standard errors)")

rf.oob_score_                  : 0.9474
5-fold CV mean (std over folds): 0.9464 (0.0067)
test accuracy p                : 0.9467  with standard error sqrt(p(1-p)/n_test) = 0.0069  (n_test = 1051)
|oob_score_ - CV mean| = 0.0009  vs  std of the forest's fold scores = 0.0067  ->  agree: True
|test accuracy - oob_score_| = 0.0006  (0.09 standard errors)
|test accuracy - CV mean|    = 0.0003  (0.05 standard errors)


**Answers Part 1**

`Out-of-bag share and out-of-bag accuracy`
- A tree sees about 63 % of the distinct training rows: the mean out-of-bag share over the 100 trees is 0.3680 against the formula (1 − 1/n)^n = 0.3678 (n = 3153), and the in-bag share is 0.6320 against 0.6322. They agree to three decimals; both are close to e^−1 = 0.3679.
- My own out-of-bag accuracy is 0.9340 and `bag.oob_score_` is 0.9340: the same number. The number of training rows that were out of bag for no tree at all is 0, so every row got at least one vote.

`(a) For the tree and for logistic regression, did bagging change the cross-validation accuracy? Judge against the paired fold differences, and give the mechanism that explains what you see.`
- Tree: yes. bag − tree = +0.0178 (std 0.0044); the bagged trees are better on 5 of 5 folds and the mean is 4.0 times the std, so it is not inside the fold spread. The forest shows the same thing: rf − tree = +0.0343 (std 0.0089), also outside the spread.
- Logistic regression: no. bag_lr − lr = -0.0006 (std 0.0016); the five differences are [0.0016, 0.0, 0.0, -0.0032, -0.0016], the mean is smaller than the std, so this is inside the fold spread. The data cannot separate bag_lr from lr, and I do not call either one better.
- Mechanism: bagging averages models fitted on different bootstrap samples, which lowers the variance of the prediction but not its bias. A fully grown tree is an unstable, high-variance learner: a different bootstrap sample gives a quite different tree, those trees make partly different mistakes, and the vote cancels a part of them. Logistic regression is a stable learner: every bootstrap sample (about 63 % of the distinct rows) gives almost the same coefficients, so the 50 members predict almost the same labels and averaging them changes almost nothing. What is left of its error is bias (a linear boundary on log-transformed features), and bagging cannot remove that.

`(b) Is the forest's out-of-bag score a fair stand-in for its cross-validation score here?`
- Yes. `oob_score_` = 0.9474 and the cross-validation mean = 0.9464; the absolute difference is 0.0009, which is not larger than the std of the forest's five fold scores (0.0067), so the two numbers are in agreement by the rule of the task.
- The test accuracy is p = 0.9467 with standard error 0.0069. Both estimates are within 0.1 standard errors of it, so all three numbers tell the same story: this forest is right on about 94.7 % of the e-mails.
- The out-of-bag score costs nothing extra (it comes from the one fit, five cross-validation fits are not needed), but each row is scored only by the roughly 37 % of the trees that did not see it, so it is a slightly different model from the full forest; on this data I see no sign that this matters.


# 2. Evaluating the filter: curves, a threshold, importances

### 2.1 ROC and precision-recall curves on the test split